[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/apartsin/DLCourseHIT/blob/main/notebooks/week05.ipynb)

# Week 5: Loss Functions & Metrics
**Introduction to Deep Learning (HIT)** &middot; Part II: Training Infrastructure

Task-appropriate losses (cross-entropy, MSE, BCE); metrics; the train and eval loop.

**Instructor practice notebook.** Run these live demonstrations during the 2-hour practice lesson. The student homework is the weekly lab.

## Goals for the practice lesson

- Choose a task-appropriate loss.
- Track metrics that reveal real performance.
- Write a clean train and evaluation loop.

## Setup
Run this first. On Colab, set Runtime > Change runtime type > GPU for the later weeks.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(0)
print('PyTorch', torch.__version__, '| device:', device)

## Demonstration 1
Run a training loop with loss and metric logging.

In [ ]:
# Training loop with loss and accuracy logging
torch.manual_seed(0)
X = torch.randn(300, 4); y = (X.sum(1) > 0).long()
model = nn.Linear(4, 2); opt = torch.optim.Adam(model.parameters(), lr=0.05); loss_fn = nn.CrossEntropyLoss()
for epoch in range(50):
    opt.zero_grad(); out = model(X); loss = loss_fn(out, y); loss.backward(); opt.step()
    if epoch % 10 == 0:
        acc = (out.argmax(1) == y).float().mean()
        print(f"epoch {epoch:2d}: loss {loss.item():.3f} acc {acc.item():.3f}")

## Demonstration 2
Show MSE on a classification task failing, then cross-entropy working.

In [ ]:
# Cross-entropy is the right loss; squaring label numbers is meaningless
logits = model(X)
ce = nn.CrossEntropyLoss()(logits, y)
mse = nn.MSELoss()(logits.argmax(1).float(), y.float())   # treats classes as numbers
print("cross-entropy (correct):", round(ce.item(), 3),
      "| MSE-on-labels (meaningless):", round(mse.item(), 3))

## Demonstration 3
Compute accuracy versus F1 on an imbalanced example.

In [ ]:
# Accuracy hides minority-class failure; F1 does not
y_true = torch.cat([torch.zeros(95), torch.ones(5)]).long()
y_pred = torch.zeros(100).long()                 # trivial: always predict class 0
acc = (y_pred == y_true).float().mean().item()
tp = ((y_pred == 1) & (y_true == 1)).sum().item()
fp = ((y_pred == 1) & (y_true == 0)).sum().item()
fn = ((y_pred == 0) & (y_true == 1)).sum().item()
prec = tp / (tp + fp + 1e-9); rec = tp / (tp + fn + 1e-9)
f1 = 2 * prec * rec / (prec + rec + 1e-9)
print(f"accuracy {acc:.2f} but F1 {f1:.2f} (recall {rec:.2f})")

---
Student materials for this week: the lab handout (`labs/week05.html`) and the curated references (`references/week05.html`) in the course site.